# Does the graft confer completion ability? (Gemma-2 9B -> 2B)

Section 5 compares the stitched recipient's completion rate to the recipient's own. The published
pair (0.151 vs 0.124) compares **across bins** -- stitched on the unsolvable bin, native on the
solvable bin -- because the unsolvable bin has no correct native first tokens to condition on.

This notebook runs both arms on **one bin**, the solvable bin, so the graft is the only thing that
varies. Three readings, increasing in strictness:

1. unconditional full-answer rate on the bin
2. completion conditional on that arm's **own** first token being right
3. **paired** -- restrict to items where **both** arms get the first token right, compare
   full-answer outcomes item by item, exact McNemar

Reading 3 is the one to quote: same problems, same first-token condition, graft toggled.

Self-contained, runs top to bottom. Setup cells are copied unchanged from
`Followup_Gemma_F26-F27-F17 (1).ipynb`; `CELL SB` and `CELL CHK` are new.

**Run cell 1, restart the kernel, then run cell 2 and everything below it.**
Set your token in the login cell first (it ships as a placeholder).

In [1]:
# === CELL 1: install (run once, then RESTART THE KERNEL, then run CELL 2) ===
import sys, subprocess, importlib.metadata as md

def _have(pkg, want=None):
    try: v = md.version(pkg)
    except md.PackageNotFoundError: return False
    return True if want is None else v == want

need = []
if not _have("transformers", "4.46.3"): need.append("transformers==4.46.3")
if not _have("accelerate"):             need.append("accelerate")
if not _have("numpy"):                  need.append("numpy<2")
if not _have("hf_transfer"):            need.append("hf_transfer")
# Only what this notebook imports. datasets / matplotlib were in the original list and nothing
# here uses them; datasets is the expensive one, since recent versions fight the transformers
# 4.46.3 pin and send pip backtracking through wheel after wheel. numpy is held below 2 so it
# cannot conflict with a pod torch built against numpy 1.x. hf_transfer is the parallel HF
# downloader -- it is enabled by an env var in CELL 2, without which downloads run single-stream.
# Add "bitsandbytes" ONLY if you set QUANTIZE_9B=True in CELL 2 (not needed at 48 GB).

if need:
    print("installing:", *need)
    subprocess.run([sys.executable, "-m", "pip", "install", *need], check=True)
else:
    print("environment already satisfies requirements -- nothing to install")

# torchvision goes BEFORE anything imports transformers: transformers lazily imports it, and a
# torch/torchvision version mismatch crashes the whole stack (the earlier L40S pod failure).
# Nothing in this notebook needs vision.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"])

# Blackwell (sm_120, e.g. RTX PRO 6000): stock torch wheels carry no sm_120 kernels and the
# first matmul dies with "no kernel image is available for execution on the device". Detect the
# card and pull the cu128 build. This REPLACES torch, so kernels differ from a run on another
# card and the last decimal can move -- CELL CHK is what says whether that drift matters.
try:
    import torch
    cap = torch.cuda.get_device_capability(0)
    print("compute capability:", cap, "| torch:", torch.__version__)
    if cap[0] >= 12 and "cu128" not in torch.__version__:
        print("Blackwell detected -> installing the cu128 torch build")
        subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "torch",
                        "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    elif cap[0] >= 12:
        print("Blackwell, cu128 torch already present")
except Exception as e:
    print("could not probe the GPU from this cell:", e)
    print("if CELL 2 prints cap (12, 0), run the cu128 install manually and restart again:")
    print("  pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128")

print("\n>>> RESTART THE KERNEL NOW, then run CELL 2 <<<")

installing: transformers==4.46.3 accelerate hf_transfer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 32.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.0/803.0 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 13.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 75.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124


/usr/local/lib/python3.11/dist-packages/torch/cuda/__init__.py:230: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 2g.48gb with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 2g.48gb GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


compute capability: (12, 0) | torch: 2.4.1+cu124
Blackwell detected -> installing the cu128 torch build
Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 161.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 402.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 169.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 195.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 308.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 430.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 466.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 539.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 245.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [1]:
# verify the stack after the restart (catch a bad resolve before any model loads)
import torch, transformers
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))
_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"gpu: {torch.cuda.get_device_name(0)} | {_gb:.1f} GB visible")
print("transformers:", transformers.__version__, "(want 4.46.3)")
# Both models in bf16 are ~5 GB (2B) + ~18.5 GB (9B) of weights, ~30 GB working with activations
# and KV cache at ARITH_BATCH=16. Under that, set QUANTIZE_9B=True in CELL 2 and add
# bitsandbytes to CELL 1.
if _gb < 32:
    print(f"  WARNING: {_gb:.1f} GB is under the ~30 GB bf16 needs -- see QUANTIZE_9B in CELL 2")
else:
    print(f"  OK: {_gb:.1f} GB is enough for both models in bf16")

torch: 2.11.0+cu128 | cuda cap: (12, 0)
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 2g.48gb | 50.9 GB visible
transformers: 4.46.3 (want 4.46.3)
  OK: 50.9 GB is enough for both models in bf16


In [2]:
# === CELL 2:imports, set_submodule shim, global config (VRAM/compute switches, primarily smoke test) ===
import os, json, math, random
# hf_transfer must be enabled BEFORE huggingface_hub is imported (transformers pulls it in), or
# the model download falls back to one slow stream. Download plumbing only; no effect on results.
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)
MODEL_2B = "google/gemma-2-2b"
MODEL_9B = "google/gemma-2-9b"

# Smoke test lever, (True for 5 min test eval False for full eval)
SMOKE_TEST = False            

# validated layers (Different for each model pair)
SINGLE_PAIR = (20, 34)
LAYER_PAIRS = [(18, 31), (20, 34), (22, 37), (24, 40)]
L2_SINGLE, L9_SINGLE = SINGLE_PAIR
PATCH_POS = -1
RIDGE_LAMBDA = 1e3

if SMOKE_TEST:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 12, 200, 120
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 20, 24, [1.0]
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
else:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 300, 3000, 2000      # ~720 unsolvable muladd
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 150, 1319, [1.0]    # full GSM8K test set
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6                # 5 seeds for the task map
    BOOT_B = 10000

ARITH_BATCH, GSM_BATCH, MAX_NEW_GSM, MAX_NEW_ARITH = 16, 8, 300, 8
RESULTS = {}   # everything defensible gets concentrated here and printed at the end
print("SMOKE_TEST =", SMOKE_TEST)

SMOKE_TEST = False


In [3]:
# === CELL F0: FOLLOW-UP CONFIG + helper defs (run right after CELL 2) ===
# This notebook reruns the minimal pipeline (models -> data -> states -> ONE task map) and then
# three follow-ups the first full run showed we need:
#   F26  shared map trained on the FULL answer sequence — closes EXP14's objective confound
#        (task maps were first-token-trained; free vectors were full-sequence-trained)
#   F27  digit-position donor probes — is the WHOLE answer linearly present, or just digit 1?
#   F17  iterative (INLP-style) answer-subspace erasure — EXP17's single-probe ablation removed
#        rank<=9 and conferral did not move; one probe finds A readout, not THE carrier
# Data/bins regenerate from the same seeds; the CELL 8 printout should roughly match the first
# run's unsolvable-bin size (Gemma ~638, Qwen ~568, Llama ~626; a few items of drift is fine).
TASK_SEEDS = [0]        # only task_maps[0] is needed as reference — big time saver vs 5 seeds

# --- helper defs copied from CELLs 10/11 so we can skip the full EXP2-4 eval bodies ---
def fd(x): return int(str(abs(int(round(x))))[0]) if x is not None else 0
def probe_first_digit(Xtr, ytr, Xte, yte, steps=300):
    Pw = torch.zeros(Xtr.shape[1], 10, requires_grad=True); opt = torch.optim.Adam([Pw], lr=1e-2)
    mu = Xtr.mean(0); A, Bx = Xtr-mu, Xte-mu
    for _ in range(steps): opt.zero_grad(); F.cross_entropy(A@Pw, ytr).backward(); opt.step()
    return (Bx@Pw.detach()).argmax(1)
@torch.inference_mode()
def first_token_confer(map_fn, idxs):
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i+ARITH_BATCH]
            _graft["vec"] = map_fn(X9e[sub])
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out
def full_confer(map_fn, idxs):
    vecs = list(map_fn(X9e[idxs]).cpu())
    return arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in idxs], vecs=vecs)
def map_task(i):
    W, b = task_maps[i]
    return lambda x9: (x9.to(DEVICE)-mu9d)@W + b
print("follow-up config set: TASK_SEEDS=[0]; helpers defined")


follow-up config set: TASK_SEEDS=[0]; helpers defined


In [ ]:
# === CELL 3: Hugging Face login (Gemma is gated) ===
from huggingface_hub import login
login("")

In [5]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

In [6]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, GSM8K, full-answer) ===
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook
 
def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m
 
def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2
 
# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out
 
# ---- GSM8K: validated prompt + extraction (from the Linear_gsm8k notebook) ----
import re as _re
def gsm_prompt(q):
    return (
        "Below are math problems with detailed step-by-step solutions.\n\n"
        "Problem: Natalia sold clips to 48 of her friends in April, and then she sold "
        "half as many clips in May. How many clips did Natalia sell altogether in April and May?\n"
        "Solution: Let's think step-by-step.\n"
        "1. Clips sold in April: 48\n"
        "2. Clips sold in May: 48 / 2 = 24\n"
        "3. Total clips: 48 + 24 = 72\n"
        "#### 72\n\n"
        f"Problem: {q}\n"
        "Solution: Let's think step-by-step."
    )
def gsm_extract(text):
    m = _re.search(r"####\s*(-?[\d,.]+)", text)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    m = _re.search(r"answer is\s*(-?[\d,.]+)", text, _re.IGNORECASE)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    nums = _re.findall(r"-?[\d,.]+", text)
    if nums:
        try: return float(nums[-1].rstrip(".").replace(",", ""))
        except ValueError: return None
    return None
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None
 
# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top
 
# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)
 
@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok
 
print("helpers defined")

helpers defined


In [7]:
# === CELL 6: load both models once; donor in bf16 by default (4-bit optional) ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_2B)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
 
# QUANTIZE_9B: set True only when the donor won't fit in bf16 (e.g. the original Gemma-2-9B on
# a small card). For 7-8B donors (Qwen-7B ~15GB, Llama-8B ~16GB) on a 20GB+ card, keep this
# False — bf16 is correct and avoids a serious failure mode: under 4-bit, this transformers/bnb
# build runs unquantized layers in FP16, and Qwen/Llama activations OVERFLOW fp16 (>65504) ->
# NaN logits -> argmax collapses to token 0 ('!'). bf16 has the exponent range to avoid this.
QUANTIZE_9B = globals().get("QUANTIZE_9B", False)
 
model_2b = AutoModelForCausalLM.from_pretrained(
    MODEL_2B, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
if QUANTIZE_9B:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True).eval()
else:
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
print("loaded:", model_2b.config.num_hidden_layers, "x2B layers,",
      model_9b.config.num_hidden_layers, "x9B layers |",
      "9B quantized" if QUANTIZE_9B else "9B bf16")
 
# Health check: catch a NaN/overflow blowup (the fp16-under-4bit failure) immediately, not
# 168 silently-filtered pairs later. A healthy donor tops a real word here, never token 0.
with torch.inference_mode():
    _hl = model_9b(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "9B produced NaN/Inf logits — numerical blowup. If QUANTIZE_9B=True, the donor is overflowing "
    "fp16; set QUANTIZE_9B=False to load it in bf16 (needs the VRAM but is numerically safe).")
del _hl
 
# Same-family requirement: the two models must share the TOKENIZER so positions align
# (the whole stitch grafts by position). NOTE: config.vocab_size is the *padded embedding*
# count, not the tokenizer — Qwen pads differently across sizes (0.5B=151936, 7B=152064)
# while sharing one tokenizer, so comparing config.vocab_size gives false alarms. Verify the
# tokenizer itself instead, by checking a probe string maps to identical ids under each model's
# own tokenizer. (We load one shared tokenizer, but this also catches an accidental mismatch.)
_tk2 = AutoTokenizer.from_pretrained(MODEL_2B); _tk9 = AutoTokenizer.from_pretrained(MODEL_9B)
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
assert _tk2(_probe).input_ids == _tk9(_probe).input_ids, (
    "tokenizer mismatch: the two models tokenize the same text differently, so positions won't "
    "align. This notebook requires a SAME-FAMILY pair sharing one tokenizer.")
del _tk2, _tk9

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

loaded: 26 x2B layers, 42 x9B layers | 9B bf16


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [8]:
# === CELL 8: EXP2 setup — arithmetic states, native correctness, reconstruction map ===
train = gen_arith(tokenizer, N_ARITH_TRAIN, random.Random(0))
evalp = gen_arith(tokenizer, N_ARITH_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval")

X9t, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in train]); X9t = X9t[L9_SINGLE]
X9e, nine_top = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in evalp]); X9e = X9e[L9_SINGLE]
keep = [i for i in range(len(evalp)) if nine_top[i] == evalp[i]["tok"]]
evalp = [evalp[i] for i in keep]; X9e = X9e[keep]
print(f"  kept {len(evalp)} the 9B solves")

X2t, _ = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in train]); X2t = X2t[L2_SINGLE]
X2e, two_top = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in evalp]); X2e = X2e[L2_SINGLE]
solvable = [two_top[i] == evalp[i]["tok"] for i in range(len(evalp))]
unsolv = [i for i in range(len(evalp)) if not solvable[i]]
solv = [i for i in range(len(evalp)) if solvable[i]]
print(f"  2B natively solves {len(solv)}/{len(evalp)} (unsolvable bin n={len(unsolv)})")

mu9, mu2, Wr = fit_ridge(X9t, X2t)
recon_map = (mu9.to(DEVICE), mu2.to(DEVICE), Wr.to(DEVICE))
def map_recon(x9): m9, m2, W = recon_map; return (x9.to(DEVICE)-m9)@W + m2

arith: 3000 train, 2000 eval
  kept 1687 the 9B solves
  2B natively solves 1055/1687 (unsolvable bin n=632)


In [9]:
# === CELL 9: EXP3 — train the task-supervised map, 5 seeds (the only stochastic part) ===
mu9d, mu2d = mu9.to(DEVICE), mu2.to(DEVICE)
task_maps = []
model_2b.requires_grad_(False)
for seed in TASK_SEEDS:
    torch.manual_seed(seed); random.seed(seed)
    W = Wr.clone().to(DEVICE).requires_grad_(True); b = mu2.clone().to(DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([W, b], lr=1e-3)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train)))
    try:
        for ep in range(TASK_EPOCHS):
            random.Random(seed*100+ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s+ARITH_BATCH]
                x9 = X9t[sub].to(DEVICE)
                _graft["vec"] = (x9 - mu9d) @ W + b
                ids, m = left_pad([train[k]["ids"] for k in sub], tokenizer.pad_token_id)
                lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
    finally:
        handle.remove(); _graft["vec"] = None
    task_maps.append((W.detach(), b.detach()))
    print(f"  seed {seed}: final batch CE {loss.item():.3f}")
model_2b.requires_grad_(True)

  seed 0: final batch CE 0.008


Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
          (rotary_emb): Gemma2RotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNorm((2304,), eps

In [10]:
# === CELL SB: PAIRED SAME-BIN COMPARISON — does the graft confer completion ability? ===
# The question: does the stitch hand the recipient a way to FINISH an answer, or only the
# answer's first token? The published comparison (0.151 vs 0.124) is across two bins --
# stitched on the unsolvable bin, unstitched on the solvable bin -- because the unsolvable bin
# has no correct native first tokens to condition on. This cell puts both arms on the SOLVABLE
# bin, the one bin where both conditions are defined, so the graft is the only thing that varies.
#
# Arms, all scored on `solv`:
#   native   no graft at all
#   recon    the ridge reconstruction map, as an information-preserving control
#   task     the seed-0 task map, trained on first-token cross-entropy
#
# The task map is also scored on `unsolv` so the published 0.886 / 0.127 reproduce in-session
# and every number below comes from one run.
#
# Scoring is unchanged from the rest of the suite: first token = argmax at the last prompt
# position against p["tok"]; full answer = greedy free generation parsed by _parse_first_int.
from fractions import Fraction as _Fr

def _mcnemar(a_ok, b_ok):
    """a_ok/b_ok: aligned bool lists. Returns (a_only, b_only, exact two-sided p)."""
    bb = sum(1 for x, y in zip(a_ok, b_ok) if x and not y)
    cc = sum(1 for x, y in zip(a_ok, b_ok) if y and not x)
    n = bb + cc
    if n == 0: return bb, cc, 1.0
    k = min(bb, cc)
    tail = sum(_Fr(math.comb(n, i)) for i in range(k + 1)) / _Fr(2) ** n
    return bb, cc, min(1.0, float(2 * tail))

@torch.inference_mode()
def native_first(idxs):
    """first_token_confer with the graft removed: same batching, same comparison, no hook."""
    out = []
    for i in range(0, len(idxs), ARITH_BATCH):
        sub = idxs[i:i+ARITH_BATCH]
        ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
        top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
        out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    return out

def native_full(idxs):
    return arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in idxs], vecs=None)

print(f"bins: solvable n={len(solv)}   unsolvable n={len(unsolv)}\n")
print("scoring the solvable bin (3 arms) ...")
ARMS = {}
ARMS["native"] = {"first": native_first(solv),                    "full": native_full(solv)}
ARMS["recon"]  = {"first": first_token_confer(map_recon,   solv), "full": full_confer(map_recon,   solv)}
ARMS["task"]   = {"first": first_token_confer(map_task(0), solv), "full": full_confer(map_task(0), solv)}
print("scoring the unsolvable bin (task map, reproduces the published values) ...")
UNSOLV_TASK = {"first": first_token_confer(map_task(0), unsolv), "full": full_confer(map_task(0), unsolv)}

def cond_bools(a):
    """this arm's full-answer outcomes, restricted to items where its OWN first token is right"""
    return [f for f, t in zip(a["full"], a["first"]) if t]

print("\n(1) unconditional, SOLVABLE bin")
for k, a in ARMS.items():
    print(f"    {k:7s} first {fmt(wilson_bools(a['first']))}   full {fmt(wilson_bools(a['full']))}")

print("\n(2) completion | that arm's OWN first token correct, SOLVABLE bin")
for k, a in ARMS.items():
    cb = cond_bools(a)
    print(f"    {k:7s} {fmt(wilson_bools(cb))}   (n={len(cb)})")

print("\n(3) PAIRED -- items where BOTH the native and task arms get the first token right")
both   = [i for i in range(len(solv)) if ARMS["native"]["first"][i] and ARMS["task"]["first"][i]]
nat_p  = [ARMS["native"]["full"][i] for i in both]
task_p = [ARMS["task"]["full"][i]   for i in both]
b_only, c_only, p_mc = _mcnemar(task_p, nat_p)
print(f"    n paired          = {len(both)}")
print(f"    native completes  = {fmt(wilson_bools(nat_p))}")
print(f"    task   completes  = {fmt(wilson_bools(task_p))}")
print(f"    discordant        = {b_only} task-only, {c_only} native-only,  exact McNemar p = {p_mc:.4f}")

u_cb = cond_bools(UNSOLV_TASK)
print("\n(ref) task map on the UNSOLVABLE bin, for continuity with the published numbers")
print(f"    first {fmt(wilson_bools(UNSOLV_TASK['first']))}   full {fmt(wilson_bools(UNSOLV_TASK['full']))}")
print(f"    completion | first correct = {fmt(wilson_bools(u_cb))}   (n={len(u_cb)})")

SAMEBIN = {
    "n_solv": len(solv), "n_unsolv": len(unsolv),
    "solvable_bin": {k: {"first": fmt(wilson_bools(a["first"])),
                         "full":  fmt(wilson_bools(a["full"])),
                         "completion_given_own_first": fmt(wilson_bools(cond_bools(a))),
                         "n_first_correct": int(sum(a["first"]))}
                     for k, a in ARMS.items()},
    "paired_native_vs_task": {
        "n_paired": len(both),
        "native_completes": fmt(wilson_bools(nat_p)),
        "task_completes":   fmt(wilson_bools(task_p)),
        "discordant_task_only": b_only, "discordant_native_only": c_only,
        "mcnemar_exact_p": round(p_mc, 4),
        "_read": "same problems, same first-token condition, graft toggled",
    },
    "unsolvable_bin_task": {
        "first": fmt(wilson_bools(UNSOLV_TASK["first"])),
        "full":  fmt(wilson_bools(UNSOLV_TASK["full"])),
        "completion_given_own_first": fmt(wilson_bools(u_cb)),
        "n_first_correct": len(u_cb),
    },
}
RESULTS["samebin_completion"] = SAMEBIN
print("\n" + json.dumps(SAMEBIN, indent=2))

bins: solvable n=1055   unsolvable n=632

scoring the solvable bin (3 arms) ...
scoring the unsolvable bin (task map, reproduces the published values) ...

(1) unconditional, SOLVABLE bin
    native  first 1.000 [0.996, 1.000]   full 0.128 [0.109, 0.149]
    recon   first 0.924 [0.907, 0.939]   full 0.118 [0.099, 0.138]
    task    first 0.957 [0.943, 0.968]   full 0.103 [0.086, 0.123]

(2) completion | that arm's OWN first token correct, SOLVABLE bin
    native  0.128 [0.109, 0.149]   (n=1055)
    recon   0.126 [0.107, 0.148]   (n=975)
    task    0.108 [0.090, 0.129]   (n=1010)

(3) PAIRED -- items where BOTH the native and task arms get the first token right
    n paired          = 1010
    native completes  = 0.127 [0.108, 0.149]
    task   completes  = 0.108 [0.090, 0.129]
    discordant        = 44 task-only, 63 native-only,  exact McNemar p = 0.0814

(ref) task map on the UNSOLVABLE bin, for continuity with the published numbers
    first 0.886 [0.859, 0.909]   full 0.128 [0.104

In [11]:
# === CELL CHK: reproduction + invariant checks ===
# (a) Does this session reproduce the published Gemma run? Anything flagged CHECK means this
#     session drifted from the numbers in the paper, and the comparison below is not directly
#     comparable to them. Single seed vs the paper's pooled values, so small drift is expected.
# (b) Are the structural invariants intact?
REF = {"n_solv": 1051, "n_unsolv": 638,
       "native_solv_full":   0.122, "recon_solv_first": 0.925, "recon_solv_full": 0.116,
       "task_solv_first":    0.951, "task_solv_full":   0.111,
       "task_unsolv_first":  0.886, "task_unsolv_full": 0.127}
TOL_RATE = 0.035

def _rate(bools): return float(np.mean(np.asarray(bools, bool)))
def chk(name, got, ref, tol):
    ok = abs(got - ref) <= tol
    print(f"  [{'PASS' if ok else 'CHECK'}] {name:20s} got {got:8.3f}   published {ref:8.3f}")
    return ok

print("(a) reproduction against the published Gemma run")
res = []
res.append(chk("n_solv",           len(solv),                          REF["n_solv"],          REF["n_solv"]*0.06))
res.append(chk("n_unsolv",         len(unsolv),                        REF["n_unsolv"],        REF["n_unsolv"]*0.06))
res.append(chk("native_solv_full", _rate(ARMS["native"]["full"]),      REF["native_solv_full"],  TOL_RATE))
res.append(chk("recon_solv_first", _rate(ARMS["recon"]["first"]),      REF["recon_solv_first"],  TOL_RATE))
res.append(chk("recon_solv_full",  _rate(ARMS["recon"]["full"]),       REF["recon_solv_full"],   TOL_RATE))
res.append(chk("task_solv_first",  _rate(ARMS["task"]["first"]),       REF["task_solv_first"],   TOL_RATE))
res.append(chk("task_solv_full",   _rate(ARMS["task"]["full"]),        REF["task_solv_full"],    TOL_RATE))
res.append(chk("task_unsolv_first",_rate(UNSOLV_TASK["first"]),        REF["task_unsolv_first"], TOL_RATE))
res.append(chk("task_unsolv_full", _rate(UNSOLV_TASK["full"]),         REF["task_unsolv_full"],  TOL_RATE))

print("\n(b) invariants")
# The solvable bin is DEFINED by the recipient getting the first token right natively, so the
# native first-token rate on it must be exactly 1.000. Anything else means `solv` is not the
# bin this analysis assumes and the paired comparison below is invalid.
nf = _rate(ARMS["native"]["first"])
print(f"  [{'PASS' if nf == 1.0 else 'FAIL'}] native first-token on solv = {nf:.4f}  (must be exactly 1.000 by bin definition)")

# A correct full answer should imply a correct first token. Greedy decoding is parsed
# numerically while the first token is compared as a token id, so a handful of tokenization
# edge cases are tolerable -- a large count would mean the two metrics disagree on what
# "correct" means, which would undercut every conditional rate in CELL SB.
for k, a in list(ARMS.items()) + [("task/unsolv", UNSOLV_TASK)]:
    bad = sum(1 for f, t in zip(a["full"], a["first"]) if f and not t)
    tot = int(sum(a["full"]))
    print(f"  [{'PASS' if bad <= max(3, 0.02*tot) else 'CHECK'}] {k:11s} full-correct but first-wrong: {bad} of {tot}")

# All arms must be aligned to the same item list, or the pairing in (3) is meaningless.
lens = {k: (len(a["first"]), len(a["full"])) for k, a in ARMS.items()}
aligned = all(v == (len(solv), len(solv)) for v in lens.values())
print(f"  [{'PASS' if aligned else 'FAIL'}] all solvable-bin arms aligned to n={len(solv)}: {lens}")

print(f"\n{sum(res)}/{len(res)} reproduction checks passed")

(a) reproduction against the published Gemma run
  [PASS] n_solv               got 1055.000   published 1051.000
  [PASS] n_unsolv             got  632.000   published  638.000
  [PASS] native_solv_full     got    0.128   published    0.122
  [PASS] recon_solv_first     got    0.924   published    0.925
  [PASS] recon_solv_full      got    0.118   published    0.116
  [PASS] task_solv_first      got    0.957   published    0.951
  [PASS] task_solv_full       got    0.103   published    0.111
  [PASS] task_unsolv_first    got    0.886   published    0.886
  [PASS] task_unsolv_full     got    0.128   published    0.127

(b) invariants
  [PASS] native first-token on solv = 1.0000  (must be exactly 1.000 by bin definition)
  [PASS] native      full-correct but first-wrong: 0 of 135
  [PASS] recon       full-correct but first-wrong: 1 of 124
  [PASS] task        full-correct but first-wrong: 0 of 109
  [PASS] task/unsolv full-correct but first-wrong: 0 of 81
  [PASS] all solvable-bin arms a

In [12]:
# === CELL SAVE ===
import pathlib
out = pathlib.Path("samebin_completion_gemma-2-2b.json")
out.write_text(json.dumps(RESULTS.get("samebin_completion", {}), indent=2))
print("wrote", out.resolve())

wrote /workspace/samebin_completion_gemma-2-2b.json
